# Vector Stores vs. Vector Databases: Architecture & In-Memory Benchmarking

This notebook explores the operational and architectural distinctions between lightweight embedded **Vector Stores** (e.g., FAISS, ChromaDB, InMemory) and cloud-managed **Vector Databases** (e.g., Pinecone, AstraDB, Qdrant).

### Key Architectural Concepts
1. **Vector Store**: An embedded library running in the same process memory (RAM). Optimized for sub-millisecond similarity search, unit tests, and lightweight deployments.
2. **Vector Database**: A distributed, networked database system managing vector indexing, persistent storage, horizontal scaling, RBAC, replication, and ACID/BASE guarantees.

## 1. Setup & Environment Initialization
Load environment variables and initialize local embeddings.

In [1]:
import os
import time
from typing import List, Dict, Any
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

# Initialize local HuggingFace embeddings for deterministic, offline execution
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("HuggingFaceEmbeddings initialized successfully.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_21876\3252856842.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings initialized successfully.


## 2. In-Memory Vector Store Construction & Latency Benchmark
Build an embedded FAISS index and measure query latency in milliseconds.

In [2]:
sample_corpus: List[Document] = [
    Document(page_content="LangChain provides standardized abstractions for LLMs, vector stores, and chains.", metadata={"category": "ai", "tier": "open_source"}),
    Document(page_content="FAISS is an in-memory library for dense vector similarity search developed by Meta.", metadata={"category": "database", "tier": "open_source"}),
    Document(page_content="Pinecone is a managed cloud-native vector database designed for high availability at scale.", metadata={"category": "database", "tier": "enterprise"}),
    Document(page_content="ChromaDB is an AI-native open-source embedding database for prototyping and production.", metadata={"category": "database", "tier": "open_source"}),
    Document(page_content="DataStax AstraDB provides serverless vector search built on Apache Cassandra.", metadata={"category": "database", "tier": "enterprise"}),
]

# Build in-memory FAISS index
vector_store = FAISS.from_documents(sample_corpus, embeddings)
print(f"FAISS index built with {vector_store.index.ntotal} vectors in memory.")

# Benchmark query latency
query = "What is an enterprise managed vector database?"
start_time = time.perf_counter()
results = vector_store.similarity_search(query, k=2)
latency_ms = (time.perf_counter() - start_time) * 1000

print(f"Query latency: {latency_ms:.3f} ms")
for rank, doc in enumerate(results, start=1):
    print(f"Rank {rank}: {doc.page_content} [category: {doc.metadata['category']}, tier: {doc.metadata['tier']}]")

assert len(results) == 2, "Expected exactly 2 retrieved documents"
assert latency_ms < 500.0, "In-memory latency should be sub-second"
print("Benchmark assertion passed.")

FAISS index built with 5 vectors in memory.
Query latency: 7.573 ms
Rank 1: Pinecone is a managed cloud-native vector database designed for high availability at scale. [category: database, tier: enterprise]
Rank 2: DataStax AstraDB provides serverless vector search built on Apache Cassandra. [category: database, tier: enterprise]
Benchmark assertion passed.


## 3. Metadata Filtering Evaluation
Evaluate structured metadata filtering to restrict retrieval scope.

In [3]:
enterprise_results = vector_store.similarity_search(
    "vector database architecture",
    k=2,
    filter={"tier": "enterprise"}
)

print("Enterprise Tier Filter Results:")
for doc in enterprise_results:
    print(f"- {doc.page_content} (tier: {doc.metadata.get('tier')})")
    assert doc.metadata.get("tier") == "enterprise", "Filter constraint violated"

print("Metadata filtering assertion passed.")

Enterprise Tier Filter Results:
- DataStax AstraDB provides serverless vector search built on Apache Cassandra. (tier: enterprise)
- Pinecone is a managed cloud-native vector database designed for high availability at scale. (tier: enterprise)
Metadata filtering assertion passed.


## 4. Architectural Decision Engine
A programmatic decision framework to determine optimal vector storage strategy based on workload constraints.

In [4]:
def recommend_vector_storage(
    num_vectors: int,
    is_production: bool,
    needs_realtime_crud: bool,
    budget_usd_monthly: float
) -> Dict[str, Any]:
    """Determine optimal vector storage architecture based on operational constraints."""
    if num_vectors < 50_000 and not is_production:
        return {
            "solution": "FAISS or Chroma Local",
            "type": "Embedded Vector Store",
            "rationale": "Dataset fits comfortably in RAM; zero infrastructure cost and zero operational overhead.",
            "estimated_cost": "$0.00"
        }
    elif num_vectors < 500_000 and budget_usd_monthly < 50.0:
        return {
            "solution": "Chroma Persistent or pgvector",
            "type": "Embedded Store or Self-Hosted SQL Vector DB",
            "rationale": "Cost-effective persistence with relational metadata support and moderate scale.",
            "estimated_cost": "< $30.00"
        }
    else:
        return {
            "solution": "Pinecone Serverless or DataStax AstraDB",
            "type": "Cloud-Native Managed Vector Database",
            "rationale": "High availability, automatic sharding, multi-region replication, and managed SLAs.",
            "estimated_cost": "Pay-as-you-go based on storage & read/write units"
        }

prototype_plan = recommend_vector_storage(num_vectors=10_000, is_production=False, needs_realtime_crud=False, budget_usd_monthly=0.0)
production_plan = recommend_vector_storage(num_vectors=2_000_000, is_production=True, needs_realtime_crud=True, budget_usd_monthly=200.0)

print("Prototype Scenario:", prototype_plan["solution"], "->", prototype_plan["type"])
print("Production Scenario:", production_plan["solution"], "->", production_plan["type"])

assert prototype_plan["type"] == "Embedded Vector Store"
assert production_plan["type"] == "Cloud-Native Managed Vector Database"
print("Decision engine assertion passed.")

Prototype Scenario: FAISS or Chroma Local -> Embedded Vector Store
Production Scenario: Pinecone Serverless or DataStax AstraDB -> Cloud-Native Managed Vector Database
Decision engine assertion passed.
